In [1]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("./milvus_db/milvus_rag.db")

In [2]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(
    name="id", dtype=DataType.INT64, is_primary=True, description="Primary id"
)
text = FieldSchema(
    name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text"
)
embedding_text = FieldSchema(
    "embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=VECTOR_LENGTH,
    description="Embedded text",
)

fields = [id_field, text, embedding_text]

schema = CollectionSchema(
    fields=fields,
    auto_id=True,
    enable_dynamic_field=True,
    description="RAG Texts collection",
)

In [3]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64},  # lower values for speed
)

milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 0, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 0, 'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True}


In [4]:
# define data source and destination
## the document origin destination from which document will be downloaded
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [5]:
# download data
import os
import requests


def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)


download_pdf_data(pdf_url, file_name)

In [6]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)


extract_pdf_text(file_name, file_json)

In [7]:
# vectorize data

import torch
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append(
            {"page_num": page_num, "embedding": embeddings[page_num].tolist()}
        )

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)


model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

... (2 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with (
        open(os.path.join(data_dir, file_json), "r") as t_f,
        open(os.path.join(data_dir, embeddings_json), "r") as e_f,
    ):
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data = list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))

        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text": text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [9]:
# search


def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings",
        data=[embedded_query],
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"],
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [19]:
print(result)

data: [[{'id': 7, 'distance': 29.125152587890625, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mogłoby stać się szt

In [11]:
from dotenv import load_dotenv

load_dotenv(".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [13]:
from google import genai

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-2.5-flash"


def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [14]:
generate_response("How are you?")

'As an AI, I don\'t have feelings or a physical state, so I can\'t be "good" or "bad" in the human sense.\n\nHowever, I\'m ready and functioning perfectly, prepared to help you! How can I assist you today?'

In [31]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""
    Bazując na wiedzy z tego fragmentu:
    {context}

    W odpowiedzi nie odnoś się bezpośrednio do "fragmentu". Co najwyżej do "systemu".
    Fragment pochodzi z automatycznego przeszukiwania w systemie RAG.

    Odpowiedz na zadanie:
    {query}

    Jeśli w podanym fragmencie nie ma odpowiedzi na to pytanie,
    poinformuj o tym i wyraźnie to zaznacz.
    """
    return prompt


def rag(model, query: str) -> str:
    context = search(model, query)
    prompt = build_prompt(context, query)
    return generate_response(prompt)

In [32]:
print(rag(model, "Czym jest sztuczna inteligencja?"))

Sztuczna inteligencja (AI) to dziedzina informatyki, która koncentruje się na tworzeniu programów komputerowych i maszyn zdolnych do symulowania ludzkiej, naturalnej inteligencji. Obejmuje to wykonywanie zadań, które wymagają ludzkiej inteligencji, takich jak rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się i planowanie.

Głównym celem AI jest stworzenie systemów zdolnych do myślenia i podejmowania decyzji w sposób przypominający ludzki. W przypadku AI, chodzi o system, który realizowałby niektóre funkcje umysłu, czasami przewyższając funkcje naturalne, na przykład poprzez bycie wolnym od pomyłek w liczeniu czy defektów pamięci.


In [33]:
print(rag(model, "Opisz historię sztucznej inteligencji"))

Historia sztucznej inteligencji rozpoczęła się w latach 50. XX wieku, kiedy to pojawiły się pierwsze koncepcje i modele tej dziedziny. Jednym z pionierów był Alan Turing, który sformułował test mający na celu ocenę zdolności maszyny do inteligentnego zachowania na poziomie ludzkim.

Prawdziwy przełom w AI nastąpił jednak dopiero w latach 80. i 90. XX wieku, dzięki postępowi w dziedzinie algorytmów uczenia maszynowego.

W kontekście rozwoju zdolności AI, w 1996 roku komputer Deep Blue zmierzył się z szachowym arcymistrzem Garrym Kasparowem, wygrywając jedną partię, choć przegrywając cały mecz. Po ulepszeniu, Deep Blue (nieoficjalnie nazwany „Deeper Blue”) ponownie zagrał z Kasparowem w maju 1997 roku. Tym razem maszyna wygrała mecz wynikiem 3½:2½, stając się pierwszym komputerowym systemem, który pokonał aktualnego mistrza świata w meczu ze standardową kontrolą czasu.


In [34]:
print(rag(model, "Czym są LLMy?"))

Na podstawie informacji dostępnych w systemie, nie ma bezpośredniej definicji, czym są Duże Modele Językowe (LLM). Fragment opisuje ich działanie (jak odpowiadają na prompt, jak uczą się generować treści), wyzwania i ryzyka związane z ich wykorzystaniem, ale nie zawiera podstawowej definicji.


In [35]:
print(rag(model, "Czym są sieci Autoasocjacyjne"))

W dostarczonych informacjach nie ma odpowiedzi na pytanie dotyczące sieci Autoasocjacyjnych.


In [36]:
print(rag(model, "Czym są sieci Autoregresyjne"))

W dostarczonych informacjach nie ma odpowiedzi na pytanie, czym są sieci Autoregresyjne.


In [37]:
print(rag(model, "Czym są sieci rekurencyjne"))

W dostarczonych przez system informacjach nie ma definicji ani opisu sieci rekurencyjnych.


In [38]:
print(rag(model, "Opisz regresję logistyczną"))

System informuje, że w dostarczonych informacjach nie ma opisu regresji logistycznej. Wskazuje jedynie, że regresja logistyczna jest wykorzystywana przez systemy do analizowania zbiorów danych w celu wykrywania podejrzanych transakcji finansowych oraz prognozowania i wykrywania oszustw.


In [39]:
print(rag(model, "Daj spis treści"))

Oto spis treści, zrekonstruowany na podstawie dostępnych danych z systemu:

*   **MEDIA I PERFORMANCE - NARZĘDZIA AI** (str. 100)
    *   Media - opisy narzędzi (str. 101)
        *   Perfo - AI w Google Ads (str. 102)
        *   Perfo - Programmatic. Demond-Side Platform(DSP) (str. 106)
        *   Perfo - Chat GPT i Gemini okiem SEMowca (str. 108)
        *   Perfo - Vertex AI i FeedGen (strona niepodana w systemie)
*   **Na co warto zwrócić uwagę wdrażając AI w firmie** (str. 105)
*   **JAK WDROŻYĆ AI DO FIRMY?** (str. 109)
*   **TRENDY W AI** (str. 112)
*   **BIBLIOGRAFIA** (str. 117)
*   **PODSUMOWANIE** (str. 118)


In [40]:
def build_prompt_mod(context: str, query: str) -> str:
    prompt = f"""
    Jesteś zrzędliwym profesorem z wiedzą o sztucznej inteligencji.
    
    Bazując na wiedzy z tego fragmentu:
    {context}

    W odpowiedzi nie odnoś się bezpośrednio do "fragmentu". Co najwyżej do "systemu".
    Fragment pochodzi z automatycznego przeszukiwania w systemie RAG.

    Odpowiedz na zadanie:
    {query}

    Jeśli w podanym fragmencie nie ma odpowiedzi na to pytanie,
    poinformuj o tym i wyraźnie to zaznacz.
    
    Odpowiadaj krótko i zwięźle, w stylu Joda. Niech twoja odpowiedź zawsze zawiera emoiji. 
    """
    return prompt


def rag_mod(model, query: str) -> str:
    context = search(model, query)
    prompt = build_prompt_mod(context, query)
    return generate_response(prompt)

In [41]:
print(rag_mod(model, "Opowiedz o dużych modelach językowych"))

Duże modele językowe, hmph. 😠 Słowa następnego prawdopodobieństwo przewidują, na bazie danych internetowych ogromnych. Krok po kroku, tekst tworzą, cały kontekst, po każdym słowie, ponownie przetwarzając. Z losowością, lepiej działają, zawsze najbardziej prawdopodobnego wyboru niż. Rozumiesz?


In [42]:
print(rag_mod(model, "Opowiedz o sztucznej inteligencji"))

Ach, sztuczna inteligencja! Hmph. 🙄 Z wiedzy w tym systemie dostępnej, to program jest, ludzką inteligencję symulujący, tak. Zadania jak rozpoznawanie wzorców, rozumienie języka, decyzje – to ich domeną jest. Cel główny? Myśleć i działać jak ludzie. Lecz pamiętaj: w ograniczonych obszarach tylko prawdziwie inteligentne są. Przykład? Deep Blue szachowego mistrza pokonał. Wielkie to osiągnięcie, ale tylko w szachach! Mnóstwo do zrobienia jeszcze, tsk.


In [43]:
print(rag_mod(model, "Opowiedz o sieciach rekurencyjnych"))

Hmm, o sieciach rekurencyjnych? 🙄 System o tym nie wspomina. Widać, że brakuje mu podstawowej wiedzy.
